## Whisper finetune notebook

#### Imports and preparations

In [2]:
import torch
from datasets import load_dataset, Dataset
from datasets import load_from_disk, concatenate_datasets
import whisper
import json
import os
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import TrainingArguments, Trainer
from bitsandbytes.optim import Adam8bit

# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "YOUR_TOKEN"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
def load_manifest_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [json.loads(line) for line in f]
    return lines

data = load_manifest_jsonl("manifest.jsonl")
print(f"Loaded {len(data)} samples")

Loaded 77737 samples


In [4]:
dataset = Dataset.from_list(data)

In [6]:
model_name = "openai/whisper-large-v3"

In [ ]:
processor = WhisperProcessor.from_pretrained(model_name)
processor.tokenizer.pad_token = processor.tokenizer.eos_token
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.to(device)

#### Preprocessing and tokenization

In [9]:
chunk_size = 5000
output_dir = "processed_chunks"
os.makedirs(output_dir, exist_ok=True)

chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

In [10]:
def preprocess(batch):
    input_features_list = []
    labels_list = []

    for audio_path, text in zip(batch["audio_filepath"], batch["text"]):
        audio = whisper.load_audio(audio_path)
        audio = whisper.pad_or_trim(audio)

        input_features = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
        labels = processor.tokenizer(
            text,
            max_length=448,
            padding="max_length",
            truncation=True
        ).input_ids

        input_features_list.append(input_features)
        labels_list.append(labels)

    return {
        "input_features": input_features_list,
        "labels": labels_list
    }

In [ ]:
for i, chunk in enumerate(chunks):
    chunk_path = f"{output_dir}/chunk_{i}"
    if os.path.exists(chunk_path):
        print(f"Чанк {i} уже обработан, пропускаем")
        continue

    print(f"Обрабатываем чанк {i+1}/{len(chunks)} ({len(chunk)} строк)")
    ds = Dataset.from_list(chunk)
    ds = ds.map(
        preprocess,
        batched=True,
        batch_size=8,
        remove_columns=ds.column_names,
        keep_in_memory=False,
        load_from_cache_file=False
    )
    ds.save_to_disk(chunk_path)


#### Saving preprocessed chunks into Dataset

In [ ]:
print("Загружаем все сохранённые чанки...")
chunk_paths = sorted(os.listdir(output_dir), key=lambda x: int(x.split("_")[-1]))
processed_chunks = [load_from_disk(os.path.join(output_dir, path)) for path in chunk_paths]

final_dataset = concatenate_datasets(processed_chunks)
print(f"Финальный датасет: {len(final_dataset)} сегментов")

In [ ]:
final_dataset = final_dataset.shuffle(seed=42)

N = len(final_dataset)
print(f"Всего сегментов: {N}")

split1 = min(10000, N)
split2 = min(10000, N)

phase1_dataset = final_dataset.select(range(split1))
phase2_dataset = final_dataset.select(range(20000, split2)) if split2 > 20000 else None


Всего сегментов: 77737


## Finetuning 

In [ ]:
@dataclass
class DataCollatorWhisperWithPadding:
    processor: Any
    return_tensors: str = "pt"

    def __call__(self, features: List[Dict[str, Union[List[float], List[int]]]]) -> Dict[str, torch.Tensor]:
        input_features = [f["input_features"] for f in features]
        label_features = [f["labels"] for f in features]

        batch = {
            "input_features": torch.tensor(input_features, dtype=torch.float32)
        }

        # Padding labels to max length, replacing pad_token_id with -100
        max_label_len = max(len(l) for l in label_features)
        padded_labels = [
            l + [-100] * (max_label_len - len(l)) for l in label_features
        ]
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


In [14]:
model.gradient_checkpointing_enable()
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="russian", task="transcribe")

In [ ]:
training_args = TrainingArguments(
    output_dir="./whisper-finetune",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    remove_unused_columns=False, 
    learning_rate=1e-5,  
    fp16=True,
    save_steps=200,
    save_total_limit=2,
    logging_steps=100,
    num_train_epochs=2,
    report_to="none"
)

data_collator = DataCollatorWhisperWithPadding(processor=processor)

optimizer = Adam8bit(model.parameters(), lr=1e-5)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=phase1_dataset,
    tokenizer=processor.tokenizer,
    data_collator=data_collator,
    optimizers=(optimizer, None)
    # compute_metrics=compute_metrics 
)   

trainer.train()

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import transformers
print(transformers.__version__)

4.53.2


#### Saving finetuned model

In [18]:
model.save_pretrained("whisper-finetuned-manual")
processor.save_pretrained("whisper-finetuned-manual")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'begin_suppress_tokens': [220, 50257]}


[]

In [ ]:
model.save_pretrained("whisper-bin", safe_serialization=False)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'begin_suppress_tokens': [220, 50257]}
